# 01 — Feasibility study (week 1 go / no-go)

**Question:** with the board stuck under the table, can we tell *where* a tap happened?

Protocol:
1. Flash the `acquisition` firmware: `cd firmware && cargo run --release -p acquisition`.
2. For each zone, record one session while tapping ~30 times on that zone:
   `uv run python scripts/record.py --label z0 --seconds 30` (then `z1`, `z2`, `z3`; and `noise` with no taps).
3. Run this notebook.

**Go criterion:** above ~90 % cross-validated accuracy on 4 zones with a simple classifier.
Then repeat with board B only / A+B, and mic only / IMU only (see README §5).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from touchwood.taps import cut_windows, detect_onsets

DATA = Path("../data/raw")
FS = 16_000
WIN, PRE = 800, 80  # 50 ms window, starting 5 ms before the onset

## 1. Load sessions and segment taps

In [ ]:
X, y, sources = [], [], []
for path in sorted(DATA.glob("*/*.npz")):
    rec = np.load(path)
    board, label = path.stem.split("_")[:2]
    if label == "noise":
        continue
    onsets = detect_onsets(rec["audio"], FS)
    windows = cut_windows(rec["audio"], onsets, WIN, PRE)
    print(f"{path.parent.name}/{path.name}: {len(windows)} taps")
    X.append(windows)
    y += [label] * len(windows)
    sources += [path.name] * len(windows)

if not X:
    raise SystemExit(f"no recordings found in {DATA.resolve()}, see the protocol above")
X = np.concatenate(X).astype(np.float32)
y = np.asarray(y)
labels = sorted(set(y))
print(X.shape, {l: int((y == l).sum()) for l in labels})

## 2. Look at the data: mean waveform and spectrum per zone

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t_ms = (np.arange(WIN) - PRE) / FS * 1e3
for label in labels:
    w = X[y == label]
    w = w / (np.abs(w).max(axis=1, keepdims=True) + 1e-9)
    axes[0].plot(t_ms, np.abs(w).mean(axis=0), label=label)
    f, pxx = signal.welch(X[y == label], FS, nperseg=256, axis=1)
    axes[1].semilogy(f, pxx.mean(axis=0), label=label)
axes[0].set(xlabel="time from onset (ms)", ylabel="mean |normalised amplitude|", title="Envelope")
axes[1].set(xlabel="frequency (Hz)", ylabel="PSD", title="Spectrum")
for ax in axes:
    ax.legend()
plt.tight_layout()

In [ ]:
# One spectrogram per zone: look for differences in decay, echoes and resonances.
fig, axes = plt.subplots(1, len(labels), figsize=(4 * len(labels), 3), sharey=True)
for ax, label in zip(np.atleast_1d(axes), labels):
    f, t, s = signal.spectrogram(X[y == label][0], FS, nperseg=64, noverlap=48)
    ax.pcolormesh(t * 1e3, f, 10 * np.log10(s + 1e-9), shading="auto")
    ax.set(title=label, xlabel="ms")
np.atleast_1d(axes)[0].set_ylabel("Hz")
plt.tight_layout()

## 3. Quick classifiers

Hand-crafted features, no deep learning yet:
- log energy in 32 frequency bands (attenuation, resonances),
- log envelope over time in 2 ms steps (decay, echoes),
- peak amplitude.

Features are normalised per tap (the tap force should not matter).

In [ ]:
def features(w: np.ndarray) -> np.ndarray:
    w = w - w.mean()
    _, pxx = signal.welch(w, FS, nperseg=256)
    bands = np.log(np.array([b.mean() for b in np.array_split(pxx, 32)]) + 1e-12)
    env = np.abs(w).reshape(-1, 32).mean(axis=1)  # 2 ms steps
    env = np.log(env / (env.max() + 1e-9) + 1e-6)
    return np.concatenate([bands - bands.max(), env, [np.log(np.abs(w).max() + 1)]])


F = np.stack([features(w) for w in X])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
models = {
    "kNN (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(5)),
    "Random forest": RandomForestClassifier(300, random_state=0),
}
for name, model in models.items():
    pred = cross_val_predict(model, F, y, cv=cv)
    print(f"{name:15s} accuracy = {(pred == y).mean():.1%}")
ConfusionMatrixDisplay.from_predictions(y, pred, normalize="true", cmap="Blues")
plt.title(name)

## 4. Verdict and next steps

- [ ] Accuracy on 4 zones: …
- [ ] Which features matter (spectrum vs. envelope)? Try each subset alone.
- [ ] Robustness: train on one session, test on another day (not a random split!).
- [ ] Two boards: compare A only / B only / A+B (add the energy ratio between boards).
- [ ] IMU: once streamed by the firmware, compare mic only / IMU only / both.